<div align="center">

# Day 03: Data Pipelines and Model Calibration

**Digital Twin with Python (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/digital-twin-veltech-NB-lecture/tree/main/days/day-03) · [Lecture page](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html) · [Interactive lab](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lab.html) · [PDF notes](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/Day03_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows; they have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from a Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures, in a few seconds to a few minutes. When the notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.6), 'axes.grid': True, 'grid.alpha': 0.3})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok



# The asset: a 48 V lithium-ion battery pack under a duty cycle.
#
#   states   SoC    state of charge, 0 to 1
#            T      cell temperature in degrees Celsius
#            Q      usable capacity in Ah, fades with use
#            R      internal resistance in ohm, grows as capacity fades
#   inputs   I(t)   current demanded by the load in A, positive when discharging
#            u      cooling-fan duty, 0 to 1
#            T_amb  ambient temperature in degrees Celsius
#
# The state of charge changes over minutes, the temperature over tens of minutes and
# the capacity over months.

@dataclass
class PackParams:
    Q0: float = 60.0        # Ah, capacity when new
    R0: float = 0.045       # Ohm, internal resistance when new
    m_cp: float = 9000.0    # J/K, thermal mass of the pack
    hA0: float = 2.2        # W/K, passive convection
    hA1: float = 12.0       # W/K, extra convection at full fan
    k_age: float = 5.5e-8   # Ah lost per A-second at the reference temperature
    Ea_R: float = 3800.0    # K, Arrhenius slope of the ageing law
    T_ref: float = 298.15   # K
    fan_W: float = 25.0     # W, electrical cost of the fan at full duty

    def as_vector(self):
        return np.array([self.R0, self.m_cp, self.hA0, self.hA1])


def pack_derivatives(t, x, I, u, T_amb, p: PackParams):
    """dx/dt for x = [SoC, T, Q, R]. Pure physics, no data, no noise."""
    SoC, T, Q, R = x
    T_K = T + 273.15
    P_gen = I ** 2 * R                                    # ohmic heating, W
    P_cool = (p.hA0 + p.hA1 * u) * (T - T_amb)            # convective loss, W
    arrh = np.exp(-p.Ea_R * (1.0 / T_K - 1.0 / p.T_ref))  # ageing accelerates with heat
    dSoC = -I / (3600.0 * max(Q, 1e-6))
    dT = (P_gen - P_cool) / p.m_cp
    dQ = -p.k_age * abs(I) * arrh
    dR = p.R0 * 0.9 * (p.Q0 - Q) / p.Q0 * 1e-3            # resistance tracks capacity fade
    return np.array([dSoC, dT, dQ, dR])


def duty_cycle(t, kind="urban", seed=RANDOM_STATE):
    """Current demand in amperes as a function of time in seconds."""
    rng = np.random.default_rng(seed)
    if kind == "urban":       # stop-start: bursts of traction, regenerative braking
        base = 14 + 26 * (np.sin(2 * np.pi * t / 420.0) > 0.15)
        regen = -18 * (np.sin(2 * np.pi * t / 420.0 + 0.9) > 0.93)
        return base + regen
    if kind == "highway":     # steady high load
        return 40 + 6 * np.sin(2 * np.pi * t / 1800.0)
    if kind == "idle":
        return 8 + 0 * t
    raise ValueError(kind)



def rk4_step(fun, t, x, dt, *args):
    k1 = fun(t, x, *args)
    k2 = fun(t + dt / 2, x + dt / 2 * k1, *args)
    k3 = fun(t + dt / 2, x + dt / 2 * k2, *args)
    k4 = fun(t + dt, x + dt * k3, *args)
    return x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def simulate(p: PackParams, T_end=3600.0, dt=1.0, x0=None, cycle="urban",
             controller=None, T_amb=25.0, seed=RANDOM_STATE):
    """Roll the physical asset forward. `controller(t, x) -> fan duty` closes the loop."""
    x = np.array([0.95, T_amb + 1.0, p.Q0, p.R0]) if x0 is None else np.array(x0, float)
    n = int(T_end / dt)
    ts = np.arange(n + 1) * dt
    X = np.zeros((n + 1, 4)); X[0] = x
    U = np.zeros(n + 1); I_log = np.zeros(n + 1)
    for k in range(n):
        t = ts[k]
        I = float(duty_cycle(t, cycle, seed))
        u = 0.0 if controller is None else float(np.clip(controller(t, x), 0, 1))
        U[k] = u; I_log[k] = I
        x = rk4_step(pack_derivatives, t, x, dt, I, u, T_amb, p)
        x[0] = np.clip(x[0], 0.0, 1.0)
        X[k + 1] = x
    U[-1] = U[-2]; I_log[-1] = I_log[-2]
    return pd.DataFrame({"t": ts, "SoC": X[:, 0], "T": X[:, 1], "Q": X[:, 2],
                         "R": X[:, 3], "u": U, "I": I_log})



@dataclass
class SensorSpec:
    """Turns a true value into a reading: noise, bias, drift, quantisation, dropout and latency."""
    noise_sd: float = 0.0       # gaussian noise, in sensor units
    bias: float = 0.0           # constant offset
    drift_per_hour: float = 0.0 # slow calibration drift
    quantum: float = 0.0        # ADC quantisation step, 0 = none
    dropout: float = 0.0        # probability a sample is lost
    latency_s: float = 0.0      # transport delay


def read_sensor(t, true_values, spec: SensorSpec, rng):
    v = np.asarray(true_values, float).copy()
    v = v + spec.bias + spec.drift_per_hour * (t / 3600.0)
    if spec.noise_sd:
        v = v + rng.normal(0, spec.noise_sd, v.shape)
    if spec.quantum:
        v = np.round(v / spec.quantum) * spec.quantum
    if spec.dropout:
        v = np.where(rng.random(v.shape) < spec.dropout, np.nan, v)
    return v


print('pack model, integrator and sensors ready')

## Python step 3: pandas data frames and time

pandas holds tables with named columns and an index, and it has dedicated tools for time series &#91;[4](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#ref-4)&#93;. This step cleans a short burst of telemetry packets from the pack: It orders them, removes a duplicate, puts them on a regular time grid, fills a gap and smooths the result.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "read_sensor" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-03/NB03_data_pipelines_and_calibration.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

### A data frame from packets

In [ ]:
import pandas as pd
packets = pd.DataFrame({"ts": [0.0, 5.2, 9.8, 9.8, 20.4, 15.1, 25.0, 41.0],
                        "T":  [35.7, 35.9, 36.0, 36.0, 36.4, 36.2, 36.5, 37.1]})
print(packets)

A `DataFrame` is built from a dictionary of equally long lists, one per column. The left column of the printout is the index, here the default row numbers. Two packets share the timestamp 9.8, and the packet of 15.1 seconds arrived after the packet of 20.4 seconds.

### Sorting and removing duplicates

In [ ]:
clean = packets.sort_values("ts").drop_duplicates(subset="ts").reset_index(drop=True)
print(clean)
print("rows before and after:", len(packets), len(clean))

`sort_values` orders the rows by a column, `drop_duplicates(subset="ts")` keeps the first row of each timestamp, and `reset_index(drop=True)` renumbers the rows. Methods can be chained, each acting on the result of the previous one.

### A time index and resampling

In [ ]:
clean.index = pd.to_timedelta(clean["ts"], unit="s")
grid = clean["T"].resample("10s").mean()
print(grid.round(3))
print("missing bins:", int(grid.isna().sum()))

`pd.to_timedelta` turns seconds into time durations, and with a time index `resample("10s")` groups the readings into ten-second bins, here averaged with `mean`. A bin without any reading becomes `NaN`, short for not a number, the marker for a missing value, which `isna` detects.

### Filling gaps and smoothing

In [ ]:
filled = grid.interpolate()
smooth = filled.rolling(3, min_periods=1).mean()
print(pd.DataFrame({"filled": filled, "smooth": smooth}).round(3))

`interpolate` fills a missing value on a straight line between its neighbours. `rolling(3)` slides a window over three consecutive values, and `mean` averages each window. `min_periods=1` allows a result at the start, where fewer than three values exist. In a twin, every interpolated value should also be flagged, as the lecture explains.

**Quick check.** What does `drop_duplicates(subset="ts")` remove from the packet table?

<details><summary>Answer</summary>

Every row whose timestamp already appeared in an earlier row, here the repeated packet at 9.8 seconds.

</details>

### Exercises for Python step 3

**Exercise 3.1.** How many rows did the cleaning remove? Store `len(packets) - len(clean)` in `n_removed`.

In [ ]:
n_removed = None   # your answer

check("Exercise 3.1", n_removed, 1)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_removed = len(packets) - len(clean)

check("Exercise 3.1", n_removed, 1)

**Exercise 3.2.** Store the mean temperature of the cleaned packets, rounded to three decimals, in `T_mean`.

In [ ]:
T_mean = None   # your answer

check("Exercise 3.2", T_mean, 36.257)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
T_mean = round(float(clean["T"].mean()), 3)

check("Exercise 3.2", T_mean, 36.257)

**Exercise 3.3.** Store the interpolated value of the bin that starts at 30 seconds, the fourth value of `filled`, rounded to three decimals, in `T_30`. Hint: `filled.iloc[3]`.

In [ ]:
T_30 = None   # your answer

check("Exercise 3.3", T_30, 36.775)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
T_30 = round(float(filled.iloc[3]), 3)

check("Exercise 3.3", T_30, 36.775)

## 1. From packets to a time series

*Lecture: [From packets to a time series](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#from-packets-to-a-time-series)*

**Python move: pandas time series.** A data frame indexed by time can be sorted, deduplicated and resampled in one line each. The three cells of this section create the packet stream, pass it through a buffer class and resample it to a grid of 10 seconds with three schemes.

**From the formulas to the code.** `make_telemetry` produces the packets, each a dictionary with the timestamp `ts` in seconds, the temperature `T`, the current `I` and the state of charge `SoC`. In the line for `T`, `rng.normal(0, 0.4) - 0.6` adds the noise with $\sigma = 0.4$ and the bias $b = -0.6$ to the true temperature, which gives the reading $z_i$. In `TelemetryBuffer.push`, the test `now - pkt["ts"] > self.stale` is the rule for late packets with the reordering window `stale_after_s=120.0`, and the dictionary `rejected` counts every rejection by its reason. The functions `zoh` (zero-order hold), `lin` and `win` are the three resampling schemes, with the packet times $\tau_i$ in `df.ts` and the grid times $t_k$ in `grid`. The table prints the root mean square error (RMSE) as `RMSE_C` and the mean error $\bar{e}$ as `mean_error_C`, both against the true temperature `truth_on_grid`. A grid time without an earlier packet gets `NaN`, short for not a number, and `np.nanmean` leaves such values out.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "T_30" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-03/NB03_data_pipelines_and_calibration.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
TRUE = PackParams()
truth = simulate(TRUE, T_end=7200, dt=1.0, cycle="highway", T_amb=35.0)

def make_telemetry(truth, mean_period=5.0, jitter=2.0, dropout=0.06, duplicate=0.02,
                   out_of_order=0.05, clock_skew_s=0.0, seed=RANDOM_STATE):
    """A packet stream with the defects of real telemetry."""
    rng = np.random.default_rng(seed)
    packets, t = [], 0.0
    while t < truth.t.iloc[-1]:
        t += max(0.2, rng.normal(mean_period, jitter))
        if t >= truth.t.iloc[-1]:
            break
        i = int(np.searchsorted(truth.t.values, t))
        if rng.random() < dropout:
            continue
        pkt = dict(ts=t + clock_skew_s,
                   T=float(truth["T"].values[i] + rng.normal(0, 0.4) - 0.6),
                   I=float(truth["I"].values[i] + rng.normal(0, 0.8)),
                   SoC=float(truth["SoC"].values[i] + rng.normal(0, 0.004)))
        packets.append(pkt)
        if rng.random() < duplicate:
            packets.append(dict(pkt))
    order = np.arange(len(packets))
    swap = rng.random(len(packets)) < out_of_order
    idx = np.where(swap)[0]
    for i in idx[:-1]:
        if i + 1 < len(order):
            order[i], order[i + 1] = order[i + 1], order[i]
    return [packets[i] for i in order]

stream = make_telemetry(truth)
print(f"{len(stream)} packets over {truth.t.iloc[-1]:.0f} s "
      f"(nominal 5 s period -> {truth.t.iloc[-1]/5:.0f} expected)")
for pkt in stream[:6]:
    print("  ", {k: round(v, 3) for k, v in pkt.items()})
ts = np.array([p["ts"] for p in stream])
print(f"\nout-of-order packets : {(np.diff(ts) < 0).sum()}")
print(f"duplicate timestamps : {len(ts) - len(np.unique(ts))}")
print(f"largest gap          : {np.diff(np.sort(ts)).max():.1f} s")

**Python note.** `np.searchsorted(truth.t.values, t)` returns the position at which `t` would be inserted into the sorted array of times, the index of the first stored time that is not earlier than `t`. `dict(pkt)` makes a copy of the dictionary, so that the duplicate is a packet of its own. In the loop, `continue` skips to the next packet time when a packet is lost, and `break` leaves the loop once the time passes the end of the record. `np.where(swap)[0]` returns the positions at which `swap` is `True`, and `order[i], order[i + 1] = order[i + 1], order[i]` swaps two neighbours in one line.

In [ ]:
class TelemetryBuffer:
    """Bounded, time-ordered, deduplicating. The twin's short-term memory.

    A ring buffer is a correctness requirement here. A twin that runs for a year cannot keep
    every packet, and a process whose memory grows without bound is stopped by the operating system."""
    def __init__(self, capacity=2000, stale_after_s=120.0):
        self.cap, self.stale = capacity, stale_after_s
        self.buf = deque(maxlen=capacity); self.seen = set()
        self.rejected = dict(duplicate=0, late=0, nonfinite=0)
    def push(self, pkt, now):
        key = (round(pkt["ts"], 6),)
        if key in self.seen:
            self.rejected["duplicate"] += 1; return False
        if now - pkt["ts"] > self.stale:
            self.rejected["late"] += 1; return False
        if not all(np.isfinite(v) for v in pkt.values()):
            self.rejected["nonfinite"] += 1; return False
        self.seen.add(key); self.buf.append(pkt)
        return True
    def frame(self):
        df = pd.DataFrame(list(self.buf))
        return df.sort_values("ts").reset_index(drop=True) if len(df) else df

from collections import deque
buf = TelemetryBuffer()
now = 0.0
for pkt in stream:
    now = max(now, pkt["ts"])
    buf.push(pkt, now)
tel = buf.frame()
print(f"accepted {len(tel)} of {len(stream)} packets   rejected {buf.rejected}")
print(f"buffer holds the most recent {len(buf.buf)} (capacity {buf.cap})")

**Python note.** `deque(maxlen=capacity)` is the ring buffer: When it is full, `append` drops the oldest element. A `set` holds each key once and answers `key in self.seen` quickly, so the test for duplicates stays cheap however many packets have arrived.

In [ ]:
grid = np.arange(0, 7200, 10.0)

def zoh(df, grid, col):
    idx = np.searchsorted(df.ts.values, grid, side="right") - 1
    out = np.where(idx >= 0, df[col].values[np.clip(idx, 0, None)], np.nan)
    return out

def lin(df, grid, col):
    return np.interp(grid, df.ts.values, df[col].values, left=np.nan, right=np.nan)

def win(df, grid, col, w=30.0):
    return np.array([np.nanmean(df[col].values[(df.ts.values > g - w) & (df.ts.values <= g)])
                     if ((df.ts.values > g - w) & (df.ts.values <= g)).any() else np.nan
                     for g in grid])

truth_on_grid = np.interp(grid, truth.t.values, truth["T"].values)
res = pd.DataFrame({"t": grid, "truth": truth_on_grid,
                    "zoh": zoh(tel, grid, "T"), "linear": lin(tel, grid, "T"),
                    "window30": win(tel, grid, "T")})
err = {c: float(np.sqrt(np.nanmean((res[c] - res.truth) ** 2))) for c in ("zoh", "linear", "window30")}
bias = {c: float(np.nanmean(res[c] - res.truth)) for c in ("zoh", "linear", "window30")}
print(pd.DataFrame({"RMSE_C": err, "mean_error_C": bias}).round(4).to_string())

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(res.t / 60, res.truth, lw=2, color="#111111", label="truth")
for c, col in (("zoh", "#c0392b"), ("linear", "#2980b9"), ("window30", "#27ae60")):
    ax.plot(res.t / 60, res[c], lw=.9, color=col, label=c)
ax.set_xlim(20, 32); ax.set_xlabel("time [min]"); ax.set_ylabel("T [C]")
ax.legend(fontsize=8); ax.set_title("Three ways to put irregular packets on a regular clock")
plt.tight_layout(); plt.show()
print("\nAll three carry the sensor's -0.6 C bias. Resampling cannot remove a systematic error;")
print("it can only decide how much random error it trades for how much delay.")

**Observe and reflect.** All three schemes carry the bias of about $-0.6$ °C and differ in their random part. Which scheme would you feed to a fan controller that must react within seconds, and which to a report on the hourly mean temperature?

## 2. Spikes, gaps and drift

*Lecture: [Spikes, gaps and drift](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#spikes-gaps-and-drift)*

The first cell injects an outage, twelve spikes and a drift into the resampled series and applies the three rules, and the second cell draws the result.

**From the formulas to the code.** In `hampel`, `k=7` gives the window of 15 values, `med` is the median $m_k$ and `n_sigma=3.0` is the factor 3 of the rule. The variable `mad` already contains the factor 1.4826, so it is the median absolute deviation (MAD) scaled to a standard deviation. A flagged value is replaced by `med`, and the array `flags` records its position. At both ends of the series the window is shorter, and missing values are left out of it. `gap_report` returns the start of every run of missing values and its length up to the next valid value, and the cell leaves the gaps empty. `drift_monitor` is the rolling mean $\bar{r}_k$ of the lecture: `rolling(window, min_periods=10).mean()` averages the values that are present among the last 60 and needs at least 10 of them, and `alarm=1.5` is the band. The residual `resid` uses the true temperature as its reference, which only a simulation can supply.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "fig" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-03/NB03_data_pipelines_and_calibration.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def inject_faults(series, t, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    s = series.copy()
    s[(t > 1800) & (t < 2100)] = np.nan                              # a 5-minute comms outage
    spikes = rng.choice(np.where(np.isfinite(s))[0], 12, replace=False)
    s[spikes] += rng.choice([-1, 1], 12) * rng.uniform(6, 14, 12)     # transient spikes
    s[t > 4500] += 0.0018 * (t[t > 4500] - 4500)                      # sensor drift begins
    return s

dirty = inject_faults(res.zoh.values, grid)

def hampel(x, k=7, n_sigma=3.0):
    """Median-absolute-deviation outlier filter. Robust where a mean/std filter is not."""
    x = np.asarray(x, float); out = x.copy(); flags = np.zeros(len(x), bool)
    for i in range(len(x)):
        lo, hi = max(0, i - k), min(len(x), i + k + 1)
        w = x[lo:hi]; w = w[np.isfinite(w)]
        if len(w) < 3 or not np.isfinite(x[i]):
            continue
        med = np.median(w); mad = 1.4826 * np.median(np.abs(w - med))
        if mad > 0 and abs(x[i] - med) > n_sigma * mad:
            out[i] = med; flags[i] = True
    return out, flags

despiked, flags = hampel(dirty)

def gap_report(x, t):
    miss = ~np.isfinite(x); 
    if not miss.any():
        return []
    edges = np.diff(miss.astype(int)); starts = np.where(edges == 1)[0] + 1
    ends = np.where(edges == -1)[0] + 1
    if miss[0]: starts = np.r_[0, starts]
    if miss[-1]: ends = np.r_[ends, len(x)]
    return [(t[s], t[min(e, len(t) - 1)] - t[s]) for s, e in zip(starts, ends)]

gaps = gap_report(despiked, grid)
print(f"outliers flagged : {int(flags.sum())}")
print(f"gaps found       : {[(f'{s:.0f}s', f'{d:.0f}s') for s, d in gaps]}")

def drift_monitor(resid, t, window=60, alarm=1.5):
    """Rolling mean of the residual. Detects a slow shift that no single sample reveals."""
    r = pd.Series(resid).rolling(window, min_periods=10).mean().values
    return t[np.argmax(np.abs(r) > alarm)] if (np.abs(r) > alarm).any() else None

resid = despiked - res.truth.values
onset = drift_monitor(resid, grid)
print(f"drift alarm at   : {onset:.0f} s   (injected at 4500 s)" if onset else "no drift detected")

**Python note.** `pd.Series(resid).rolling(window, min_periods=10).mean()` computes for every position the mean of the last `window` values, and `min_periods=10` returns a value as soon as ten of them exist. `np.argmax(condition)` returns the first position at which the condition is true. In `gap_report`, `~` turns every `True` into `False` and back, so `miss` marks the missing readings, and `miss.astype(int)` turns the marks into ones and zeros, so that `np.diff` gives 1 where a gap begins and -1 where it ends.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.4))
ax.plot(grid / 60, res.truth, lw=2, color="#111111", label="truth")
ax.plot(grid / 60, dirty, lw=.8, color="#e67e22", label="received (spikes, gap, drift)")
ax.plot(grid / 60, despiked, lw=1.1, color="#2980b9", label="after Hampel filter")
ax.scatter(grid[flags] / 60, dirty[flags], s=22, color="#c0392b", zorder=5, label="flagged")
if onset:
    ax.axvline(onset / 60, ls="--", lw=1, color="#8e44ad", label="drift alarm")
ax.axvline(4500 / 60, ls=":", lw=1, color="#8e44ad")
ax.set_xlabel("time [min]"); ax.set_ylabel("T [C]"); ax.legend(fontsize=8, ncol=2)
ax.set_title("Three pathologies, three responses")
plt.tight_layout(); plt.show()
print("Spikes are replaced and flagged. Gaps are listed and left empty. Drift raises an alarm.")
print("A twin that compensates for drift silently hides the failing sensor until it fails completely.")

The next cell prints the time of the drift alarm and the delay after the start of the drift at 4500 s.

In [ ]:
alarm_t = float(onset)
print(f"drift alarm at {alarm_t:.0f} s, {(alarm_t - 4500) / 60:.0f} minutes after the drift began")

**Observe and reflect.** The alarm comes 24 minutes after the drift began. Change the band `alarm=1.5` of `drift_monitor` to 1.0 and to 2.0 and run the section again. What does a narrower band gain, and which risk does it bring for a healthy sensor?

## 3. Calibration with least squares &#91;[1](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#ref-1)&#93;

*Lecture: [Calibration](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#calibration)*

**From the formulas to the code.** `forward(theta, obs)` is the simulation $\hat{y}_k(\theta)$ and returns the temperature at every measurement time. It advances the model of the pack with the Runge-Kutta step `rk4_step` of Day 2 in steps of `DT_CAL = 10` seconds, with the recorded current and fan duty of each step. `residuals` returns simulation minus measurement, the negative of the residual $r_k$ of the lecture, which makes no difference once the values are squared. `least_squares` minimises half the sum of squares, so its result `sol.x` is the estimate $\hat{\theta}$, and `np.sqrt(np.mean(sol.fun ** 2))` is $\sqrt{S / n}$, the RMSE of the fit. `theta0` is the datasheet guess, `BOUNDS` holds the lower and the upper limit of each parameter, and `xtol`, `ftol` and `max_nfev` are the two tolerances and the cap on evaluations. `sol.nfev` counts the simulations of the solver itself, without the additional ones that measure the slopes.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "alarm_t" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-03/NB03_data_pipelines_and_calibration.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
from scipy.optimize import least_squares, differential_evolution
import time

DT_CAL = 10.0    # the twin is calibrated on 10-second telemetry, not on the 1-second truth

def make_observations(p_true, T_end=5400.0, noise=0.35, cycle="highway", seed=RANDOM_STATE):
    """A calibration experiment: run the pack with a fan schedule that excites all parameters,
    then downsample to the rate at which telemetry arrives."""
    rng = np.random.default_rng(seed)
    sched = lambda t, x: 0.0 if (t % 1800) < 900 else 0.85     # fan on for half of each 30 min
    d = simulate(p_true, T_end=T_end, dt=1.0, cycle=cycle, T_amb=35.0, controller=sched)
    d = d.iloc[::int(DT_CAL)].reset_index(drop=True)
    d["T_meas"] = d["T"].values + rng.normal(0, noise, len(d))
    return d

obs = make_observations(TRUE)
print(f"calibration run: {obs.t.iloc[-1]:.0f} s of data at {DT_CAL:.0f} s resolution "
      f"= {len(obs)} samples")
print("fan duty alternates 0 / 0.85 every 15 min, which is what makes hA1 identifiable")

THETA_NAMES = ["R0", "m_cp", "hA0", "hA1"]
THETA_TRUE = np.array([TRUE.R0, TRUE.m_cp, TRUE.hA0, TRUE.hA1])
BOUNDS = np.array([[0.02, 0.09], [5000., 15000.], [1.0, 5.0], [4.0, 25.0]])

def forward(theta, obs):
    p = PackParams(R0=theta[0], m_cp=theta[1], hA0=theta[2], hA1=theta[3])
    x = np.array([obs.SoC.iloc[0], obs["T"].iloc[0], p.Q0, theta[0]])
    n = len(obs); T = np.empty(n); T[0] = x[1]
    tv, Iv, uv = obs.t.values, obs.I.values, obs.u.values
    for k in range(n - 1):
        x = rk4_step(pack_derivatives, tv[k], x, DT_CAL, Iv[k], uv[k], 35.0, p)
        x[0] = min(max(x[0], 0.0), 1.0); T[k + 1] = x[1]
    return T

def residuals(theta, obs):
    return forward(theta, obs) - obs.T_meas.values

t0 = time.time()
theta0 = np.array([0.038, 10200.0, 2.9, 8.0])      # the datasheet guess from Day 1
sol = least_squares(residuals, theta0, bounds=(BOUNDS[:, 0], BOUNDS[:, 1]),
                    args=(obs,), xtol=1e-9, ftol=1e-9, max_nfev=300)
t_lsq = time.time() - t0

est = pd.DataFrame({"true": THETA_TRUE, "datasheet": theta0, "identified": sol.x},
                   index=THETA_NAMES)
est["err_before_%"] = 100 * (est.datasheet - est.true) / est.true
est["err_after_%"] = 100 * (est.identified - est.true) / est.true
print(est.round(4).to_string())
print(f"\nleast_squares: {sol.nfev} function evaluations, {t_lsq:.1f} s, "
      f"final RMSE {np.sqrt(np.mean(sol.fun ** 2)):.4f} C")

**Python note.** `least_squares(residuals, theta0, ..., args=(obs,))` receives the function `residuals`, which returns the vector of residuals for one parameter set, and squares and adds them itself. `args=(obs,)` passes the data as the second argument of `residuals`, and the comma makes `(obs,)` a tuple of one element.

**Observe and reflect.** All four estimates are about 5 percent too small, and yet the RMSE of the fit equals the noise of the sensor. Before reading the next section: Which property of the equation of the pack could allow this?

## 4. A hidden symmetry

*Lecture: [A hidden symmetry](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#a-hidden-symmetry)*

A global optimiser &#91;[6](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#ref-6)&#93; and a second experiment without the fan, then a direct test of the symmetry.

**From the formulas to the code.** `sse` is the objective $S(\theta)$, printed as SSE, the sum of squared errors. `differential_evolution` receives the same bounds: `popsize=5` gives 5 candidates per parameter, 20 in all, `maxiter=12` is the number of generations, and `polish=True` adds the final local search. For the second experiment the column `u` is set to 0, so the product $hA_1 u$ vanishes from the equation. The second fit may use at most 200 evaluations, `max_nfev=200`, and the cell prints how many it used. The last cell multiplies all four parameters by `k = 1.5`, the factor $\lambda$ of the lecture, and compares the two temperature curves. `ratios` computes the three identifiable ratios for the truth and for each optimiser.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "est" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-03/NB03_data_pipelines_and_calibration.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def sse(theta, obs):
    return float(np.sum(residuals(theta, obs) ** 2))

# --- 1. the well-posed problem
t0 = time.time()
de = differential_evolution(sse, bounds=list(map(tuple, BOUNDS)), args=(obs,),
                            maxiter=12, popsize=5, tol=1e-5, seed=RANDOM_STATE, polish=True)
t_de = time.time() - t0
print("Fan exercised: four parameters")
print(pd.DataFrame({"true": THETA_TRUE, "least_squares": sol.x, "diff_evolution": de.x},
                   index=THETA_NAMES).round(4).to_string())
print(f"  least_squares    {sol.nfev:5d} evals  {t_lsq:5.1f} s   SSE {sse(sol.x, obs):9.2f}")
print(f"  diff_evolution   {de.nfev:5d} evals  {t_de:5.1f} s   SSE {de.fun:9.2f}")
print("\n  Both optimisers reach almost the same SSE with very different parameter values.")
print("  The next cell shows why.")

# --- 2. the ill-posed problem: fan never on, so hA1 never affects the output
obs_bad = make_observations(TRUE).copy()
obs_bad["u"] = 0.0
obs_bad["T"] = simulate(TRUE, T_end=5400, dt=1.0, cycle="highway",
                        T_amb=35.0)["T"].values[::int(DT_CAL)]
rng = np.random.default_rng(1)
obs_bad["T_meas"] = obs_bad["T"].values + rng.normal(0, 0.35, len(obs_bad))

# With the fan off, hA1 has no effect on any simulated temperature, so the optimiser finds no slope
# in that direction. The option max_nfev limits the number of evaluations in any case.
sol_bad = least_squares(residuals, theta0, bounds=(BOUNDS[:, 0], BOUNDS[:, 1]),
                        args=(obs_bad,), xtol=1e-8, ftol=1e-8, max_nfev=200)
print("\nIll-posed problem (fan never switched on)")
print(pd.DataFrame({"true": THETA_TRUE, "identified": sol_bad.x},
                   index=THETA_NAMES).round(4).to_string())
print(f"  fit RMSE {np.sqrt(np.mean(sol_bad.fun ** 2)):.4f} C  <- an excellent fit")
print(f"  the solver used {sol_bad.nfev} of at most 200 evaluations")
print(f"  but hA1 is off by {100*(sol_bad.x[3]-THETA_TRUE[3])/THETA_TRUE[3]:+.0f}%, "
      f"and the data carry no information about it.")

In [ ]:
# The temperature equation depends on R0, hA0 and hA1 only through their ratios to m_cp.
k = 1.5
T_a = forward(THETA_TRUE, obs)
T_b = forward(k * THETA_TRUE, obs)
print(f"all four parameters scaled by {k}: largest change of the temperature trajectory {np.abs(T_a - T_b).max():.2e} C\n")
ratios = lambda th: pd.Series({"R0/m_cp": th[0] / th[1], "hA0/m_cp": th[2] / th[1], "hA1/m_cp": th[3] / th[1]})
cmpr = pd.DataFrame({"true": ratios(THETA_TRUE), "least_squares": ratios(sol.x), "diff_evolution": ratios(de.x)})
cmpr["lsq_err_%"] = 100 * (cmpr.least_squares - cmpr.true) / cmpr.true
cmpr["de_err_%"] = 100 * (cmpr.diff_evolution - cmpr.true) / cmpr.true
print("the identifiable combinations:")
print(cmpr.to_string(float_format=lambda v: f"{v:.3e}" if abs(v) < 0.01 else f"{v:.2f}"))

**Observe and reflect.** Scale all four parameters with `k = 0.5` instead of 1.5. Does the largest change of the trajectory stay at the level of the rounding error? Which experiment other than a temperature measurement could reveal the size of the pack?

## 5. Breaking the symmetry with an independent measurement

*Lecture: [A hidden symmetry](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#a-hidden-symmetry)*

**From the formulas to the code.** `M_CP_MEASURED = 9000.0 * 1.02` is the heat capacity from the separate test, 2 percent above the true value. `residuals3` inserts it into the parameter vector, so the optimisers adjust only `R0`, `hA0` and `hA1`, in this order in `th3`. Each result equals the ratio of section 4 times `M_CP_MEASURED`. `THETA_FIT` joins the three estimates with the measured heat capacity and is the calibrated parameter set of the Kalman filter and of the physics check in section 7.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "cmpr" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-03/NB03_data_pipelines_and_calibration.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
# A separate heat-capacity test gives m_cp with an uncertainty of about 2 percent.
M_CP_MEASURED = 9000.0 * 1.02
def residuals3(th3, o):
    return forward(np.array([th3[0], M_CP_MEASURED, th3[1], th3[2]]), o) - o.T_meas.values
B3 = BOUNDS[[0, 2, 3]]
t0 = time.time()
sol3 = least_squares(residuals3, theta0[[0, 2, 3]], bounds=(B3[:, 0], B3[:, 1]), args=(obs,), xtol=1e-9, ftol=1e-9, max_nfev=300)
t_lsq3 = time.time() - t0
de3 = differential_evolution(lambda th, o: float(np.sum(residuals3(th, o) ** 2)), bounds=list(map(tuple, B3)), args=(obs,),
                             maxiter=12, popsize=5, tol=1e-5, seed=RANDOM_STATE, polish=True)
fix = pd.DataFrame({"true": THETA_TRUE[[0, 2, 3]], "least_squares": sol3.x, "diff_evolution": de3.x}, index=["R0", "hA0", "hA1"])
fix["lsq_err_%"] = 100 * (fix.least_squares - fix.true) / fix.true
fix["de_err_%"] = 100 * (fix.diff_evolution - fix.true) / fix.true
print(f"m_cp fixed at {M_CP_MEASURED:.0f} J/K from the separate test\n")
print(fix.round(4).to_string())
print(f"\nleast_squares {sol3.nfev} evaluations in {t_lsq3:.1f} s; differential evolution {de3.nfev} evaluations")
THETA_FIT = np.array([sol3.x[0], M_CP_MEASURED, sol3.x[1], sol3.x[2]])

**Exercise A.** With `m_cp` fixed, what value of `hA1` does least squares identify? Store it in `hA1_fit`.

In [ ]:
hA1_fit = None   # your answer

check("Exercise A", hA1_fit, float(sol3.x[2]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
hA1_fit = float(sol3.x[2])

check("Exercise A", hA1_fit, float(sol3.x[2]))

**Observe and reflect.** The heat capacity of the separate test is 2 percent too high, and every fitted parameter comes out about 2 percent too high as well. How would a test result 5 percent too low change the three estimates?

## 6. Validation on an unseen drive cycle

*Lecture: [Validation](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#validation)*

**From the formulas to the code.** `holdout` is the unseen hour on the urban cycle, and `np.sqrt(np.mean(r**2))` is the RMSE between simulation and measurements on it. The cell tests `theta0`, the datasheet guess, and `sol.x`, the four-parameter fit of section 3. `r_fit` holds the residuals of the calibration data, again as simulation minus measurement. The mean and the two input correlations therefore carry the opposite sign of those of the residual $r_k$ of the lecture. `np.corrcoef(a, b)[0, 1]` is the correlation coefficient $\rho$: `lag1` pairs `r_fit[:-1]` with `r_fit[1:]`, every residual with the next one, and `corr_I` and `corr_u` pair the residuals with the current and with the fan duty.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "hA1_fit" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-03/NB03_data_pipelines_and_calibration.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
# --- hold-out on a different duty cycle
holdout = make_observations(TRUE, T_end=3600.0, cycle="urban", seed=3)

for nm, th in (("datasheet", theta0), ("identified", sol.x)):
    r = forward(th, holdout) - holdout.T_meas.values
    print(f"{nm:11s} hold-out (urban cycle) RMSE {np.sqrt(np.mean(r**2)):.3f} C   "
          f"max |err| {np.abs(r).max():.3f} C")

# --- residual structure
r_fit = residuals(sol.x, obs)
lag1 = float(np.corrcoef(r_fit[:-1], r_fit[1:])[0, 1])
corr_I = float(np.corrcoef(r_fit, obs.I.values)[0, 1])
corr_u = float(np.corrcoef(r_fit, obs.u.values)[0, 1])
print(f"\nresiduals: mean {r_fit.mean():+.4f} C   sd {r_fit.std():.4f} C "
      f"(sensor sd was 0.350)")
print(f"           lag-1 autocorrelation {lag1:+.3f}   corr with I {corr_I:+.3f}   "
      f"corr with fan {corr_u:+.3f}")

fig, ax = plt.subplots(1, 3, figsize=(12, 3))
ax[0].plot(obs.t / 60, r_fit, lw=.6, color="#2980b9"); ax[0].axhline(0, color="grey", lw=.6)
ax[0].set_title("residual vs time"); ax[0].set_xlabel("min")
ax[1].scatter(obs.I, r_fit, s=3, alpha=.3, color="#c0392b")
ax[1].set_title("residual vs current"); ax[1].set_xlabel("I [A]")
ax[2].hist(r_fit, bins=40, color="#27ae60", alpha=.8)
ax[2].set_title("residual distribution")
plt.tight_layout(); plt.show()

**Observe and reflect.** The residuals of the fit look like noise: no trend over time and no slope against the current. Sketch what the left panel would show if the model missed a heat source that grows with the current.

## 7. A Kalman filter, and a physics check of the sensor &#91;[2](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#ref-2), [3](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#ref-3)&#93;

*Lecture: [A Kalman filter instead of overwriting](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#a-kalman-filter-instead-of-overwriting)*

**From the formulas to the code.** In `kalman_shadow`, `x[1]` is the belief $\hat{T}_k$ and `P` is its variance, and the arguments `q` and `r` are the variances $Q$ and $R$. The call of `rk4_step` and the line `P = P + q` are the prediction, `nu` is the innovation $\nu_k$, `K = P / (P + r)` is the gain, and the line after it corrects the belief and the variance. The variable `P` is overwritten in place, so it holds $P_k^{-}$ after the prediction and $P_k$ after the correction. At the first reading there is no prediction, and the filter starts from the first true temperature with `P = 1.0`. `rmse_overwrite` is the RMSE of the raw readings against the true temperature, the error of a belief that copies every reading. In the second cell, `pred = forward(THETA_FIT, obs)` is the prediction of the physics check, which uses no reading, `rolling(30, min_periods=30).mean()` is the average $\bar{r}_k$ over 30 residuals, and `np.abs(...) > 1.0` is the alarm.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "corr_u" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-03/NB03_data_pipelines_and_calibration.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def kalman_shadow(o, theta, q=0.02, r=0.35 ** 2):
    """Scalar Kalman filter on the temperature: predict with the calibrated model, correct with each reading."""
    p = PackParams(R0=theta[0], m_cp=theta[1], hA0=theta[2], hA1=theta[3])
    x = np.array([o.SoC.iloc[0], o["T"].iloc[0], p.Q0, theta[0]]); P = 1.0
    T_hat, innovations = [], []
    for k in range(len(o)):
        if k > 0:
            x = rk4_step(pack_derivatives, o.t.values[k - 1], x, DT_CAL, o.I.values[k - 1], o.u.values[k - 1], 35.0, p)
            P = P + q                                   # predict: uncertainty grows
        nu = o.T_meas.values[k] - x[1]                  # innovation: reading minus prediction
        K = P / (P + r)                                 # Kalman gain
        x[1] = x[1] + K * nu; P = (1 - K) * P           # correct: uncertainty shrinks
        T_hat.append(x[1]); innovations.append(nu)
    return np.array(T_hat), np.array(innovations)

T_kf, nu = kalman_shadow(obs, THETA_FIT)
rm = lambda e: float(np.sqrt(np.mean(np.asarray(e) ** 2)))
rmse_overwrite = rm(obs.T_meas.values - obs["T"].values)
rmse_kalman = rm(T_kf - obs["T"].values)
print(f"overwrite with each reading: RMSE {rmse_overwrite:.3f} C")
print(f"Kalman filter              : RMSE {rmse_kalman:.3f} C")
plt.figure(figsize=(9, 3.2))
plt.plot(obs.t / 60, obs.T_meas, ".", ms=2, color="#bbbbbb", label="readings")
plt.plot(obs.t / 60, obs["T"], color="#2c3e50", lw=1.2, label="truth")
plt.plot(obs.t / 60, T_kf, color="#2980b9", lw=1.8, label="Kalman belief")
plt.xlabel("time (min)"); plt.ylabel("temperature (C)"); plt.legend(fontsize=8); plt.show()

The next cell computes by which factor the Kalman filter reduces the error of the belief.

In [ ]:
kf_gain = rmse_overwrite / rmse_kalman
print(f"the Kalman filter divides the error of the belief by {kf_gain:.1f}")

**Observe and reflect.** Run `kalman_shadow(obs, THETA_FIT, q=0.5)` and `kalman_shadow(obs, THETA_FIT, q=0.002)` in a new cell and compare their RMSE with the settled gains of the lecture. Why can a very small `q` be dangerous on a real pack, although it is the best choice here?

In [ ]:
# A sensor bias of -3 C appears after 45 minutes.
o_bias = obs.copy(); o_bias["T_meas"] = obs.T_meas.values + np.where(obs.t.values >= 2700, -3.0, 0.0)
_, nu_b = kalman_shadow(o_bias, THETA_FIT)
print(f"Kalman innovation mean before {nu_b[obs.t.values < 2700].mean():+.3f} C, after {nu_b[obs.t.values >= 2700].mean():+.3f} C")
print("The filter follows the biased readings, so its innovations hardly move.\n")
pred = forward(THETA_FIT, obs)                         # the calibrated model, run without any reading
check_res = pd.Series(o_bias.T_meas.values - pred).rolling(30, min_periods=30).mean()
alarm = np.abs(check_res.values) > 1.0
first = obs.t.values[np.argmax(alarm)] if alarm.any() else None
print(f"independent physics check: false alarms before the fault {int(alarm[obs.t.values < 2700].sum())}, "
      f"first alarm at {first:.0f} s (fault from 2700 s)")
plt.figure(figsize=(9, 2.8)); plt.plot(obs.t / 60, check_res, color="#27ae60", label="reading minus model, 5-minute mean")
plt.axhline(1, color="#c0392b", ls="--", lw=1); plt.axhline(-1, color="#c0392b", ls="--", lw=1)
plt.xlabel("time (min)"); plt.ylabel("C"); plt.legend(fontsize=8); plt.show()

**Observe and reflect.** The physics check needs a model that runs without readings. What happens to this check after many hours, when the small errors of the calibrated parameters have added up, and how could the twin restart its prediction without trusting the suspect sensor?

## 8. Your asset: calibration with a measured heat input

*Lecture: [Your asset](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#your-asset)*

Change `ASSET` to `server_rack`, `motor` or `greenhouse` and run the cell again.

**From the formulas to the code.** The inner function `simulate` of `calibrate_asset` is the explicit Euler step of the asset model. Its line `T[k + 1] = T[k] + dt * (...) / C_` adds the time step times the right-hand side of the equation, with `asset["power"](t[k])` as the known heat input $P_{\mathrm{gen}}(t_k)$. The cooling duty `u` is drawn from 0, 0.5 and 1 and repeated for 20 steps. The optimiser works on the logarithms of the three parameters and `np.exp(th)` turns them back, so that no parameter can become negative. `start` holds the starting values, and every simulation of the fit starts from the first reading `y[0]`.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "first" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-03/NB03_data_pipelines_and_calibration.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-8":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
ASSET = "battery"        # choose: battery, server_rack, motor or greenhouse

ASSETS = {
    "battery":     dict(name="battery pack", C=9000.0, hA0=2.2, hA1=12.0, T_amb=35.0, T0=36.0, limit=46.0,
                        horizon=7200.0, dt=5.0, power=lambda t: (40 + 6 * np.sin(2 * np.pi * t / 1800)) ** 2 * 0.045),
    "server_rack": dict(name="server rack", C=1.2e5, hA0=40.0, hA1=400.0, T_amb=24.0, T0=26.0, limit=35.0,
                        horizon=7200.0, dt=10.0, power=lambda t: 3000 + 1500 * np.maximum(0, np.sin(2 * np.pi * t / 3600))),
    "motor":       dict(name="motor winding", C=4000.0, hA0=1.5, hA1=8.0, T_amb=30.0, T0=32.0, limit=90.0,
                        horizon=5400.0, dt=5.0, power=lambda t: (20 + 8 * np.sin(2 * np.pi * t / 1200)) ** 2 * 0.3),
    "greenhouse":  dict(name="greenhouse air", C=3.0e6, hA0=500.0, hA1=4000.0, T_amb=18.0, T0=20.0, limit=32.0,
                        horizon=86400.0, dt=60.0, power=lambda t: 30000 * np.maximum(0, np.sin(2 * np.pi * t / 86400))),
}

def run_asset(a, controller=None, params=None, sensor_sd=0.0, seed=0, T_end=None, dt=None):
    """Explicit Euler for dT/dt = (P(t) - (hA0 + hA1 u)(T - T_amb)) / C; the controller maps a reading to a duty u."""
    p = dict(a); p.update(params or {}); dt = dt or a["dt"]; n = int((T_end or a["horizon"]) / dt)
    t = np.arange(n) * dt; T = np.empty(n); u = np.zeros(n); rng = np.random.default_rng(seed); T[0] = a["T0"]
    for k in range(n - 1):
        reading = T[k] + sensor_sd * rng.normal()
        u[k] = controller(reading) if controller else 0.0
        T[k + 1] = T[k] + dt * (a["power"](t[k]) - (p["hA0"] + p["hA1"] * u[k]) * (T[k] - p["T_amb"])) / p["C"]
    return t, T, u


from scipy.optimize import least_squares

def calibrate_asset(asset, seed=RANDOM_STATE):
    """Noisy readings under a changing cooling duty, then a fit of C, hA0 and hA1 with the heat input measured."""
    rng = np.random.default_rng(seed); dt = asset["dt"]; n = int(asset["horizon"] / dt)
    t = np.arange(n) * dt; u = np.repeat(rng.choice([0.0, 0.5, 1.0], n // 20 + 1), 20)[:n]
    def simulate(C_, h0, h1, T0):
        T = np.empty(n); T[0] = T0
        for k in range(n - 1):
            T[k + 1] = T[k] + dt * (asset["power"](t[k]) - (h0 + h1 * u[k]) * (T[k] - asset["T_amb"])) / C_
        return T
    truth = simulate(asset["C"], asset["hA0"], asset["hA1"], asset["T0"])
    y = truth + rng.normal(0, 0.02 * (asset["limit"] - asset["T_amb"]), n)
    start = np.log([asset["C"] * 1.5, asset["hA0"] * 0.6, asset["hA1"] * 1.4])
    fit = least_squares(lambda th: simulate(*np.exp(th), y[0]) - y, start)
    return t, u, y, simulate(*np.exp(fit.x), y[0]), np.exp(fit.x)

asset = ASSETS[ASSET]
t_a, u_a, y_a, fit_a, est_a = calibrate_asset(asset)
true_a = np.array([asset["C"], asset["hA0"], asset["hA1"]])
calib_asset = pd.DataFrame({"true": true_a, "estimate": est_a, "error_%": 100 * (est_a / true_a - 1)}, index=["C", "hA0", "hA1"])
print(f"{asset['name']}: the heat input is measured, so all three parameters can be identified")
print(calib_asset.round(3).to_string())
scale = 3600 if asset["horizon"] > 20000 else 60
fig, ax = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(t_a / scale, y_a, ".", ms=2, alpha=.4, label="readings"); ax[0].plot(t_a / scale, fit_a, color="#c0392b", label="calibrated model")
ax[0].legend(fontsize=8); ax[0].set_ylabel("temperature (C)"); ax[1].step(t_a / scale, u_a, where="post", color="#7f8c8d")
ax[1].set_ylabel("cooling duty"); ax[1].set_xlabel("time (h)" if scale == 3600 else "time (min)"); plt.tight_layout(); plt.show()

**Python note.** The fit works on the logarithms of the parameters: `start = np.log([...])`, and `np.exp(th)` turns them back. A parameter written as the exponential of another number can never become negative, and a step on the logarithm changes every parameter by a percentage instead of a fixed amount.

**Exercise B.** What is the largest absolute error, in percent, among the three estimates for your asset? Store it in `worst_error`.

In [ ]:
worst_error = None   # your answer

check("Exercise B", worst_error, float(calib_asset["error_%"].abs().max()))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
worst_error = float(calib_asset["error_%"].abs().max())

check("Exercise B", worst_error, float(calib_asset["error_%"].abs().max()))

**Observe and reflect.** The heat input of these assets is measured, so all three parameters are identifiable. Which measurement would the battery pack need to reach the same, and why is it harder to obtain for a battery than for a server rack?

## 9. Going further (optional): rugged landscapes, sensitivity and bootstrap &#91;[7](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#ref-7)&#93;

*Lecture: [Going further (optional)](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lecture.html#going-further-optional)*

**From the formulas to the code.** `rugged` is the test function $f(a, b)$ of the lecture, with the abbreviations `u = a - 2` and `v = b + 1`. `minimize(..., method="L-BFGS-B")` is the local optimiser, a method that follows the slope of the function and is named after Broyden, Fletcher, Goldfarb and Shanno, in a version with limited memory (L) and bounds (B). `local_sensitivity` multiplies one parameter by `1 + rel` with `rel=0.02` and returns the root mean square change $s_j$ of the simulated temperatures. In `bootstrap_theta`, `res_pool` holds the residuals of the fit, `rng.choice(res_pool, len(res_pool), replace=True)` draws the new residuals with replacement, and each refit starts at `sol.x`. `np.percentile(B, 2.5, axis=0)` and `np.percentile(B, 97.5, axis=0)` are the two ends of the 95 percent interval.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "worst_error" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-03/NB03_data_pipelines_and_calibration.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-9":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def rugged(z):
    """A smooth bowl with a periodic ripple phase-locked to its centre.

    The ripple vanishes exactly at (2, -1), so that point is the unique global minimum with
    f = 0. Everywhere else the ripple creates local minima to which a gradient method
    converges and which it reports as success."""
    a, b = z
    u, v = a - 2.0, b + 1.0
    bowl = 0.6 * (u ** 2 + v ** 2)
    ripple = 3.0 * (np.sin(2.2 * u) ** 2 + np.sin(2.2 * v) ** 2)
    return bowl + ripple

from scipy.optimize import minimize
BR = [(-6.0, 8.0), (-8.0, 6.0)]

starts = [(-5.0, 5.0), (-2.0, 3.0), (5.0, -6.0), (7.0, 4.0)]
rows = []
for st in starts:
    m = minimize(rugged, st, method="L-BFGS-B", bounds=BR)
    rows.append(dict(start=str(st), grad_x=round(m.x[0], 2), grad_y=round(m.x[1], 2),
                     grad_f=round(float(m.fun), 3), grad_nfev=m.nfev))
d = differential_evolution(rugged, BR, maxiter=60, popsize=15, seed=RANDOM_STATE, tol=1e-8)
gp = pd.DataFrame(rows)
print(gp.to_string(index=False))
print(f"\ndifferential_evolution -> ({d.x[0]:.2f}, {d.x[1]:.2f})  f = {d.fun:.3f}  "
      f"in {d.nfev} evaluations")
print(f"designed global minimum -> (2.00, -1.00)  f = {rugged([2.0, -1.0]):.3f}")

A, B_ = np.meshgrid(np.linspace(-6, 8, 400), np.linspace(-8, 6, 400))
Z = np.array([[rugged((a, b)) for a in A[0]] for b in B_[:, 0]])
i, j = np.unravel_index(Z.argmin(), Z.shape)
print(f"brute-force grid minimum -> ({A[0][j]:.2f}, {B_[:, 0][i]:.2f})  f = {Z.min():.3f}"
      f"   [400 x 400 = {Z.size} evaluations, a cost that grows exponentially with the dimension]")
fig, ax = plt.subplots(figsize=(5.4, 4.6))
ax.contourf(A, B_, np.log1p(Z), levels=40, cmap="viridis")
for st, r in zip(starts, rows):
    ax.plot([st[0], r["grad_x"]], [st[1], r["grad_y"]], "-o", ms=4, lw=1, color="#ffffff")
ax.plot(d.x[0], d.x[1], "*", ms=18, color="#e74c3c", label="differential evolution")
ax.plot(2, -1, "x", ms=12, mew=2.5, color="#f1c40f", label="true global minimum")
ax.legend(fontsize=8, loc="upper left")
ax.set_xlabel("parameter 1"); ax.set_ylabel("parameter 2")
ax.set_title("Four gradient descents (white) from four starting points")
plt.tight_layout(); plt.show()

In [ ]:
def local_sensitivity(theta, obs, rel=0.02):
    """Change of the output when each parameter is raised by the fraction rel. Near zero means not identifiable."""
    base = forward(theta, obs)
    out = {}
    for j, nm in enumerate(THETA_NAMES):
        th = theta.copy(); th[j] *= (1 + rel)
        out[nm] = float(np.sqrt(np.mean((forward(th, obs) - base) ** 2)))
    return pd.Series(out)

s_good = local_sensitivity(THETA_TRUE.copy(), obs)
s_bad = local_sensitivity(THETA_TRUE.copy(), obs_bad)
sens = pd.DataFrame({"fan exercised": s_good, "fan never on": s_bad})
sens["ratio"] = sens["fan exercised"] / sens["fan never on"].replace(0, np.nan)
print("output change [C] for a +2% change in each parameter\n")
print(sens.round(5).to_string())
print("\nThe sensitivity of hA1 collapses to zero without fan actuation. The analysis costs four")
print("simulations and shows before any fit which parameters the data can support.")

In [ ]:
# --- bootstrap confidence intervals on the identified parameters
def bootstrap_theta(n_boot=10, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    base = forward(sol.x, obs); res_pool = obs.T_meas.values - base
    out = []
    for b in range(n_boot):
        o = obs.copy()
        o["T_meas"] = base + rng.choice(res_pool, len(res_pool), replace=True)
        s = least_squares(residuals, sol.x, bounds=(BOUNDS[:, 0], BOUNDS[:, 1]),
                          args=(o,), xtol=1e-7, ftol=1e-7, max_nfev=40)
        out.append(s.x)
    return np.array(out)

B = bootstrap_theta()
ci = pd.DataFrame({
    "true": THETA_TRUE, "estimate": sol.x,
    "ci_lo": np.percentile(B, 2.5, axis=0), "ci_hi": np.percentile(B, 97.5, axis=0),
}, index=THETA_NAMES)
ci["rel_width_%"] = 100 * (ci.ci_hi - ci.ci_lo) / ci.estimate
ci["covers_truth"] = (ci.ci_lo <= ci.true) & (ci.true <= ci.ci_hi)
print(f"95% bootstrap intervals from {len(B)} refits\n")
print(ci.round(4).to_string())
print("\nThe relative width belongs in the report. A parameter with a 40% interval")
print("should not be reported as a single number with four decimal places.")
print(f"\nintervals covering the true value: {int(ci.covers_truth.sum())} of {len(ci)}")

**Explore.** Move the controls and watch the result update. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W
def explore(q=0.02, r_sd=0.35, bias=0.0):
    o = obs.copy(); o["T_meas"] = obs.T_meas.values + np.where(obs.t.values >= 2700, bias, 0.0)
    Th, _ = kalman_shadow(o, THETA_FIT, q=q, r=r_sd ** 2)
    res = pd.Series(o.T_meas.values - forward(THETA_FIT, obs)).rolling(30, min_periods=30).mean()
    fig, ax = plt.subplots(1, 2, figsize=(11, 3))
    ax[0].plot(obs.t / 60, obs["T"], label="truth"); ax[0].plot(obs.t / 60, Th, label="Kalman belief"); ax[0].legend(fontsize=8)
    ax[1].plot(obs.t / 60, res, color="#27ae60"); ax[1].axhline(1, ls="--", color="r"); ax[1].axhline(-1, ls="--", color="r")
    ax[1].set_title("physics check residual"); plt.tight_layout(); plt.show()
    print(f"belief RMSE {rm(Th - obs['T'].values):.3f} C")
W.interact(explore, q=W.FloatSlider(value=0.02, min=0.0005, max=0.5, step=0.0005, readout_format=".4f"),
           r_sd=W.FloatSlider(value=0.35, min=0.05, max=2.0, step=0.05), bias=W.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.5));

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Implement the telemetry buffer as a ring buffer of fixed memory and show that its output is identical to the version of the notebook. |
| Electronics and Communication Engineering | Add packet reordering with a delay distribution of your choice and measure how large the reordering window of the buffer must be. |
| Electrical and Electronics Engineering | Write the equation of a resistor-capacitor (RC) circuit that is fed by a current source and find its structural symmetry, if any, when only the voltage is measured. |
| Mechanical Engineering | For a mass-spring-damper measured by its position only, decide which of mass, stiffness and damping are identifiable when the force is known and when it is not. |
| Biomedical Engineering | In a one-compartment drug model measured by concentration, show that dose and volume are not separately identifiable without a known dose. |
| Civil Engineering | Design a calibration experiment for the thermal model of a room with a heater and a window, so that both conductances become identifiable. |
| Aeronautical Engineering | Tune the process noise of the Kalman filter so that the belief tracks a fast thermal transient without importing the sensor noise, and report the trade-off. |
| Biotechnology | Calibrate a logistic growth model to noisy biomass data and report bootstrap intervals for the growth rate and the carrying capacity. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-03/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/digital-twin-veltech-NB-lecture/tree/main/days/day-03).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>